# Top 10 — แคตตาล็อก (Pediatric vs Adult)

โน้ตบุ๊กนี้รวม **ตาราง + กราฟ** สำหรับหัวข้อด้านล่าง โดยใช้ไฟล์ parquet เดียวกับ `top_five_faers_terms_pediatric_vs_adult.ipynb`

**ความถี่:** ใช้ **จำนวนรายงานไม่ซ้ำ** (`safetyreportid`) ต่อคีย์ (report-level) สำหรับทุกตารางด้านล่าง

**การจัดอันดับ “intersect”:** ใช้เฉพาะคีย์ที่มีทั้งใน Pediatric และ Adult (inner join) แล้วเรียงตาม **`min(n_ped, n_adu)`** เพื่อเน้นรายการที่ **แข็งแรงในทั้งสองกลุ่ม** — จากนั้นเลือก **10 อันดับแรก**

---

<details>
<summary><strong>สารบัญ (คลิกเพื่อเปิด/ปิดแต่ละหัวข้อ)</strong></summary>

1. **Top 10 drug (Pediatric และ Adult แยกกัน)** — `rxcui` อันดับในแต่ละกลุ่ม (ไม่ใช่ intersect)
2. **Top 10 คู่ drug–ADR (Pediatric และ Adult แยกกัน)** — `(rxcui, reaction_meddrapt)` หลัง dedup ต่อรายงาน
3. **Top 10 intersect ADR** — `reaction_meddrapt` ที่มีในทั้งสองกลุ่ม เรียงตาม `min(n_ped, n_adu)`
4. **Top 10 intersect drug** — `rxcui` ที่มีในทั้งสองกลุ่ม เรียงตาม `min(n_ped, n_adu)` (ป้าย FAERS + RxNorm จาก Pediatric)
5. **Top 10 intersect คู่ drug–ADR** — `(rxcui, reaction_meddrapt)` ที่มีในทั้งสองกลุ่ม เรียงตาม `min(n_ped, n_adu)` (ป้ายจาก Pediatric)

</details>

<details>
<summary><strong>1) Top 10 drug — Pediatric / Adult แยกกัน</strong></summary>

- **คีย์:** `rxcui` (ไม่รวมปฏิกิริยา)
- **นับ:** distinct `safetyreportid` ต่อ `rxcui`
- **ป้าย:** ชื่อ FAERS ที่พบบ่อย + `ingredient` (RxNorm)

</details>

<details>
<summary><strong>2) Top 10 คู่ drug–ADR — Pediatric / Adult แยกกัน</strong></summary>

- **คีย์:** `(rxcui, reaction_meddrapt)` หลัง `drop_duplicates` ต่อ `(safetyreportid, rxcui, reaction_meddrapt)`
- **นับ:** จำนวนรายงานไม่ซ้ำต่อคู่
- **ป้าย:** `medicinal_product(RxNorm) − PT`

</details>

<details>
<summary><strong>3) Top 10 intersect ADR</strong></summary>

- **คีย์:** `reaction_meddrapt`
- **เงื่อนไข:** PT นั้นต้องปรากฏในทั้ง Pediatric และ Adult
- **อันดับ:** `min(n_reports_ped, n_reports_adu)` จากสูงไปต่ำ

</details>

<details>
<summary><strong>4) Top 10 intersect drug (`rxcui`)</strong></summary>

- **คีย์:** `rxcui`
- **เงื่อนไข:** มีรายงานในทั้งสองกลุ่ม
- **อันดับ:** `min(n_ped, n_adu)` จากสูงไปต่ำ
- **ป้าย:** สร้างจากข้อมูล Pediatric (โหมด FAERS + RxNorm ต่อ rxcui)

</details>

<details>
<summary><strong>5) Top 10 intersect คู่ drug–ADR</strong></summary>

- **คีย์:** `(rxcui, reaction_meddrapt)`
- **เงื่อนไข:** คู่เดียวกันปรากฏในทั้งสองกลุ่ม
- **อันดับ:** `min(n_ped, n_adu)` จากสูงไปต่ำ
- **ป้าย:** สร้างจากข้อมูล Pediatric สำหรับคู่นั้น

</details>

---

**โค้ด (แหล่งเดียว):** `_top10_catalog_mega.py` — เซลล์ด้านล่าง `exec` ไฟล์นั้นแล้วเรียก `run_all(...)`

**วิธีรัน:** รันเซลล์โหลดข้อมูลก่อน แล้วรันเซลล์เดียวที่รวมสคริปต์ + `run_all`


In [ ]:
from pathlib import Path

import pandas as pd

try:
    from IPython.display import display
except ImportError:
    display = print

ROOT = Path("..").resolve()
OUT = ROOT / "data" / "output"
PATHS = {
    "Pediatric": OUT / "Pediatric" / "patient_report_reporter_drug_reaction_full_data.parquet",
    "Adult": OUT / "Adult" / "patient_report_reporter_drug_reaction_full_data.parquet",
}
for name, p in PATHS.items():
    if not p.exists():
        raise FileNotFoundError(f"Missing {name}: {p}")

print("Loading parquet (Adult may take a while)...")
df_ped = pd.read_parquet(PATHS["Pediatric"])
df_adu = pd.read_parquet(PATHS["Adult"])
ped_count = df_ped["safetyreportid"].nunique()
adu_count = df_adu["safetyreportid"].nunique()
print("OK:", {k: str(v) for k, v in PATHS.items()})
print("Pediatric rows:", len(df_ped), "| Adult rows:", len(df_adu))
print("Distinct reports — Ped:", ped_count, "| Adult:", adu_count)


In [ ]:
from pathlib import Path

# โหลด helper + กราฟทั้งหมดจากไฟล์เดียว (แก้ที่ _top10_catalog_mega.py)
_p = Path(".").resolve() / "_top10_catalog_mega.py"
if not _p.is_file():
    raise FileNotFoundError(f"Expected {_p}")
exec(_p.read_text(encoding="utf-8"), globals())
run_all(df_ped, df_adu, ped_count, adu_count)
